# 语音识别（机械臂主机运行）

采集链路：USB 麦克风 → Vosk 离线中文识别 → 关键词“帮我分类”命中 → 输出 `VoiceCommand`（`voice_code=1`，SORT_ONE）。

## 依赖说明（重要）

板子**没有外网 DNS**，`pip install vosk` 会报 `Temporary failure in name resolution`，所以不要用在线安装单元。
本 notebook 会优先从工程根目录里的离线包安装：

```
vosk-0.3.45-py3-none-manylinux2014_aarch64.whl
srt-3.5.3.tar.gz          # vosk 的依赖，必须一起装
```

`PyAudio` 该内核环境已自带（0.2.12），无需安装。依次运行下面的单元即可。

In [ ]:
# 环境准备：定位工程、按需离线安装 vosk + srt、导入检查
from pathlib import Path
import sys, subprocess

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_SRC = next((p for p in candidates if (p / "voice" / "models" / "vosk-cn").is_dir()), None)
if PROJECT_SRC is None:
    raise SystemExit("未找到 voice/models/vosk-cn，请把 voice/ 整个目录放到当前目录下")
sys.path.insert(0, str(PROJECT_SRC))
print("PROJECT_SRC =", PROJECT_SRC)


def ensure_package(import_name, patterns):
    try:
        module = __import__(import_name)
        print(f"{import_name}: 已安装", getattr(module, "__version__", ""))
        return module
    except ModuleNotFoundError:
        pass
    files = []
    for pattern in patterns:
        files.extend(sorted(PROJECT_SRC.glob(pattern)))
    if not files:
        raise SystemExit(f"{import_name} 未安装，且工程目录里没有离线包 {patterns}")
    for item in files:
        print("离线安装", item.name)
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                        "--no-build-isolation", str(item)], check=True)
    return __import__(import_name)


vosk = ensure_package("vosk", ["vosk-*-aarch64.whl"])
ensure_package("srt", ["srt-*.tar.gz"])
pyaudio = ensure_package("pyaudio", [])
print("python", sys.version.split()[0], sys.executable)

In [ ]:
# 模型完整性校验 + 试加载（能打印 MODEL_OK 才说明模型可用）
import vosk

MODEL_DIR = PROJECT_SRC / "voice" / "models" / "vosk-cn"
need = [
    "am/final.mdl", "conf/mfcc.conf", "graph/HCLr.fst", "graph/Gr.fst",
    "graph/phones/word_boundary.int", "ivector/final.ie", "ivector/final.dubm",
]
missing = [name for name in need if not (MODEL_DIR / name).exists()]
print("模型目录:", MODEL_DIR)
print("缺失文件:", missing or "无")
assert not missing, "Vosk 模型不完整，请重新上传 voice/models/vosk-cn"

from voice.voice_command import VOICE_CODES
from voice.recognizer import DEFAULT_GRAMMAR

print("命令映射:", VOICE_CODES)
print("受限语法:", DEFAULT_GRAMMAR)
model = vosk.Model(str(MODEL_DIR))
print("MODEL_OK")

In [ ]:
# 麦克风自检：列出输入设备，并实际采 1 秒音频计算音量（RMS 应大于 0）
import pyaudio, audioop

pa = pyaudio.PyAudio()
print("默认输入设备索引:", pa.get_host_api_info_by_index(0).get("defaultInputDevice"))
for i in range(pa.get_device_count()):
    info = pa.get_device_info_by_index(i)
    if info.get("maxInputChannels", 0) > 0:
        print(f"[{i}] {info['name']}  channels={info['maxInputChannels']}  rate={int(info['defaultSampleRate'])}")

RATE = 16000
stream = pa.open(format=pyaudio.paInt16, channels=1, rate=RATE, input=True,
                 input_device_index=None, frames_per_buffer=1024)
data = b""
for _ in range(int(RATE / 1024)):
    data += stream.read(1024, exception_on_overflow=False)
stream.stop_stream()
stream.close()
pa.terminate()

rms = audioop.rms(data, 2)
print(f"采样 1 秒完成，RMS = {rms}")
if rms == 0:
    print("⚠️ 采样全静音：检查麦克风接线、是否被静音，或改 input_device_index 指定上面的某个设备")

In [ ]:
# 实时识别：跑 40 秒，对着麦克风说「帮我分类」
import subprocess, time, json

cmd = [sys.executable, "-m", "voice.app", "--mic", "--asr", "vosk",
       "--vosk-model", str(MODEL_DIR), "--grammar", "--debug", "--cooldown-s", "3"]
print(" ".join(cmd), flush=True)

proc = subprocess.Popen(cmd, cwd=str(PROJECT_SRC), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
lines = []
deadline = time.time() + 40
try:
    while time.time() < deadline:
        line = proc.stdout.readline()
        if not line:
            break
        line = line.rstrip()
        lines.append(line)
        if line:
            print(line, flush=True)
finally:
    proc.kill()
    proc.wait(timeout=5)

commands = [json.loads(x) for x in lines if x.startswith("{")]
print(f"\n命中命令 {len(commands)} 条")
for item in commands:
    print(item.get("text"), "voice_code=", item.get("voice_code"), "task_id=", item.get("task_id"))

In [ ]:
# 无麦克风时验证链路（直接喂文本，验证关键词匹配与冷却）
from voice.recognizer import MockTextBackend, VoiceRecognizer
from voice.command_gate import VoiceCommandGate
from voice.voice_command import VoiceCommand, VOICE_CODES
import time

items = [("帮我分类", 0.95), ("今天天气不错", 0.9), ("帮我分类", 0.95)]
rec = VoiceRecognizer(MockTextBackend(items), min_confidence=0.5)
gate = VoiceCommandGate(cooldown_s=3.0, min_confidence=0.5)
now = time.time()
for utt in rec.backend:
    name = VoiceRecognizer.match_command(utt.text)
    print("识别:", utt.text, "conf=", utt.confidence, "命中:", name)
    if name is not None:
        cmd = VoiceCommand(text=utt.text, confidence=utt.confidence, voice_code=VOICE_CODES[name])
        emitted = gate.update(cmd, now)
        now += 1.0
        print("  下发:", emitted.to_json() if emitted else "被冷却/门限拦截")

## 输出格式

```json
{"text":"帮我分类","confidence":0.93,"command_id":"vc_xxx","voice_code":1,"source":"voice","captured_at":"2026-11-13T16:20:00","latency_ms":0.0,"task_id":null}
```

C 侧读取 `voice_code == 1`（SORT_ONE）执行一次分拣，并补全 `task_id`。

## 常见问题

- `No module named vosk` → 跑第 2 个单元，从工程目录的离线 wheel 安装（本板无外网 DNS，在线 pip 必失败）。
- 安装报 `No module named srt` → vosk 的 `__init__.py` 顶层依赖 srt，同单元已一起离线安装。
- 一直识别不到 → 先看第 4 个单元的 RMS；RMS 为 0 说明没采到声音，检查麦克风或换 `input_device_index`。
- 能听到声音但识别成别的词 → 已启用 `--grammar` 受限语法，只会解码“帮我分类/开始分拣/暂停/复位/取消/急停”；仍不准则靠近麦克风、放慢语速。
- 反复触发 → 冷却默认 3 秒，可用 `--cooldown-s` 调整。